# Trader Workflow

*Consolidated handover of the three-notebook pipeline: **01 Trader Metrics**, **02 Trader Controls**, **03 Trader Execution**. Spot crypto on Binance. No shorting, no margin, no leverage. The single question this pipeline exists to answer is whether there is a rule the model can trade that clears fees on data it has never seen.*

*This is a technical handover. It states what each stage does, why it is built that way, and how to run it, and it carries the key code and figures from the three notebooks so a new reader can pick up the repository and continue the search. Read Overview first, then the three walkthroughs in order.*

## 0. Overview

### 0.1 After-Fee Evaluation

The goal is one trading strategy that makes money after fees on data it has never seen. Every stage in these three notebooks serves that test, and nothing is accepted without passing it.

A candidate is scored on a blind final year that is held out from all training and tuning. It must clear three bars at once: beat a coin flip at the real base rate, beat buy and hold, and return more than zero per trade after the round-trip fee of about 0.20 percent. A rule that ranks trades well but still loses to the fee is a NO-GO. The fee is the adversary, not the market.

The search runs across three decision frames so intraday, interday, and longer holding each get a fair test. A frame is a whole system: its own bars, its own label, its own features, scored on its own out-of-sample. The frames are compared, never pooled, because a five-minute label and a four-hour label do not mean the same thing.

### 0.2 Pipeline Overview

The work splits into three notebooks that run in order. Each hands a clean input to the next.

| Notebook | Role | The question it answers | Key outputs |
| --- | --- | --- | --- |
| **01 Trader Metrics** | Signal layer | What is worth measuring on a coin | indicators, entry and exit signals, a ranked shortlist, the decision checklist |
| **02 Trader Controls** | Risk and rules layer | What the account is allowed to do | position caps, stops, the daily loss circuit, the fee model, the four-gate screen, position sizing |
| **03 Trader Execution** | Modeling and search layer | Whether a tradeable edge survives fees | the datasets, the features, the model, and the after-fee scoreboard that returns GO or NO-GO |

Metrics defines the vocabulary of signals. Controls sets the boundaries any strategy must live inside, above all the fee and the loss limits. Execution builds the datasets and the model and runs the honest test. The controls are not a step bolted on at the end. They are the filter the whole search is judged against.

### 0.3 Data Frames

Each frame is built from the same Binance Vision archives into its own dataset, `dataset_5m_allmarket`, `dataset_1h_allmarket`, and `dataset_4h_allmarket`, all stored as Parquet. One builder serves all three and retunes its feature windows, its label horizon, and its liquidity and volatility screens to the frame.

| Frame | Bar | Trade style it targets | Default label |
| --- | --- | --- | --- |
| **5m** | 5 minutes | intraday scalps | a shorter ATR barrier, roughly two hours |
| **1h** | 1 hour | day-to-day swings | +2 / -1 ATR triple barrier, two wall-clock days |
| **4h** | 4 hours | multi-day holds | +2 / -1 ATR triple barrier, two wall-clock days |

The higher-timeframe context shifts with the frame, so a five-minute model still sees the one-hour and four-hour trend, a one-hour model sees the four-hour and daily trend, and a four-hour model sees the daily and weekly trend. That is how each frame gets multi-resolution information without the datasets ever being merged. Every modelling section in Part 3 runs this same 1h, 4h, 5m loop and reports one block per frame.

### 0.4 Hard Rules

Every candidate strategy lives inside a fixed set of controls, summarised here and detailed in Part 2. None is negotiable, and the after-fee test in the last row overrides any in-sample result.

| Control | Setting | Why |
| --- | --- | --- |
| Position cap | 5 percent of equity at entry | one bad name cannot sink the book |
| Default size | half Kelly, a quarter on minimum signals | size to the edge, not the conviction |
| Fat-pitch exception | one position to 10%, with reward-to-risk > 3:1, a named cause and exit | rare, documented, reversible |
| Hard stop | exit at 7 percent below cost basis | trend protection, provisional and under review |
| Trailing stop | 10 percent from the peak on a winner | let winners run, cap the give-back, provisional |
| Daily circuit | halt new orders if rolling 24-hour drawdown passes 3 percent | stop the bleeding, existing stops stay live |
| Drawdown ramp | below a 5 percent rolling-week loss, cut new size with each further 1 percent | shrink the book, do not just block it |
| Cash floor | keep at least 10 percent in cash | always able to act |
| Position limit | at most 3 new positions per week | forces selectivity |
| Direction | spot only, never short, never margin, never leverage | the mandate |
| Averaging down | never | a loser is exited or held, not fed |
| Anchoring | cost basis never enters hold or sell logic | decide on forward value only |
| The bar | nothing ships unless it beats a coin flip, buy-and-hold, and the fee out of sample | the fee is the adversary |

### 0.5 Running It

The notebooks run on a MacPorts Python through a project `.venv` kernel. Secrets live in the run environment, never in the repository: the Binance and Alpaca keys, `BINANCE_TESTNET`, and the single money switch `LIVE_TRADING`, which stays false until a strategy has earned the right to trade. No real order is placed while it is false; the agent reads the market or runs against the Binance testnet.

One knob drives the modelling cells. `LOAD_FRAME` selects the active frame, 1h by default. Import Data loads that frame's Parquet into `df` and `feat`, and every modelling cell consumes those two. The multi-frame sections in Part 3, model assessment, stability, edge diagnostics, selectivity, and regime conditioning, loop over 1h, 4h, and 5m on their own and restore the active frame when they finish, so the rest of the notebook is unaffected.

Run order is top to bottom. The code cells carried into this document have their outputs cleared, so run them on the Mac with the `.venv` kernel to regenerate every figure and table. One environment caveat: the repository sits on an exFAT volume that scatters `._` AppleDouble files, which can crash matplotlib; clear them if an import throws a `0xb0` decode error.

## 1. Trader Metrics

*The signal layer, from `01-trader-metrics`. What the pipeline measures on each coin before any model is trained. Subsections to draft, in order:*
- *1.1 Environment Setup*
- *1.2 Rank Universe*
- *1.3 Performance Metrics*
- *1.4 Entry Points*
- *1.5 Exit Points*
- *1.6 Exit Geometry*
- *1.7 Decision Checklist*

## 2. Trader Controls

*The risk and rules layer, from `02-trader-controls`. The boundaries every candidate must live inside. Subsections to draft, in order:*
- *2.1 Key Variables*
- *2.2 Volatility Filter*
- *2.3 Four-Gate Screen*
- *2.4 Edge Sizing*
- *2.5 Hold Period*
- *2.6 Stacked Defences*
- *2.7 Signal Journal*
- *2.8 Frozen Harness*

## 3. Trader Execution

*The modeling and search layer, from `03-trader-execution`. Where datasets, features, and model meet the after-fee test, run per frame. Subsections to draft, in order:*
- *3.1 Data Preparation*
- *3.2 Survivorship Pipeline*
- *3.3 Feature Variables*
- *3.4 Entries Exits*
- *3.5 Sharpening Entries*
- *3.6 Training Regime*
- *3.7 Model Tuning*
- *3.8 Model Assessment*
- *3.9 Stability*
- *3.10 Edge Diagnostics*
- *3.11 Selectivity Test*
- *3.12 Regime Conditioning*

## 4. Shared Method

*To draft: the machinery shared by every section, the multi-frame loop and why frames are never pooled, the after-fee scoreboard and the GO / NO-GO gate, embargoed time-series splits, and the provenance discipline that keeps the picture and the numbers from drifting.*

## 5. Findings

*To draft: the honest state of the search. Where each frame lands against the after-fee bar, the one signal that has shown promise (cross-sectional relative strength), and where the edge search points next.*

## Appendices

*To draft:*
- *A. File Map: which module powers which section*
- *B. Glossary: label, triple barrier, ATR, Supertrend, Kelly, embargo, base rate, RMSEratio, and the rest*
- *C. Environment: MacPorts, the `.venv` kernel, the exFAT caveat, and the environment variables*